In [1]:
import arcpy
import os

# --- 用户设置 ---
# 请将此路径修改为您包含 SHP 文件的根目录
root_directory = r"D:\510182彭州市\510182彭州市"
# -----------------

print("脚本开始...")
print("将在以下目录中搜索SHP文件 (包括所有子目录): " + root_directory)

# 1. 使用 arcpy.da.Walk 遍历目录和子目录
try:
    for root, dirnames, filenames in arcpy.da.Walk(root_directory, datatype="FeatureClass"):
        
        # filenames 是一个包含所有要素类名称的列表
        for f in filenames:
            
            # 2. 检查文件名是否以 .shp 结尾 (确保它是一个shapefile)
            if not f.lower().endswith(".shp"):
                continue 

            shp_path = os.path.join(root, f)
            print("---------------------------------")
            print("正在处理: " + shp_path)

            try:
                # 3. 检查 'BSM' 字段是否存在且类型为 'Double'
                field_list = arcpy.ListFields(shp_path)
                bsm_field = None
                
                # 遍历字段 (使用 .upper() 忽略大小写)
                for field in field_list:
                    if field.name.upper() == "BSM":
                        bsm_field = field
                        break
                
                # 如果字段存在且类型正确，则开始转换
                if bsm_field and bsm_field.type == "Double":
                    print("  找到 'BSM' (Double) 字段。开始转换...")
                    
                    # 定义一个临时的字段名
                    temp_field_name = arcpy.ValidateFieldName("BSM_TEMP", shp_path)
                    
                    # 步骤 A: 添加一个新的 LONG 类型临时字段
                    print("  步骤 1/4: 添加临时字段 '" + temp_field_name + "' (LONG)...")
                    arcpy.AddField_management(shp_path, temp_field_name, "LONG")
                    
                    # 步骤 B: 将 'BSM' (Double) 的值计算到新字段 (Long)
                    # 注意：这将截断小数部分
                    print("  步骤 2/4: 计算值 (从 BSM 到 " + temp_field_name + ")...")
                    
                    # *** ArcGIS Pro 关键更改: 使用 "PYTHON3" ***
                    arcpy.CalculateField_management(shp_path, temp_field_name, "!BSM!", "PYTHON3")
                    
                    # 步骤 C: 删除旧的 'BSM' (Double) 字段
                    print("  步骤 3/4: 删除旧的 'BSM' (Double) 字段...")
                    arcpy.DeleteField_management(shp_path, "BSM")
                    
                    # 步骤 D: 将临时字段重命名回 'BSM'
                    print("  步骤 4/4: 重命名 '" + temp_field_name + "' 为 'BSM'...")
                    arcpy.AlterField_management(shp_path, temp_field_name, "BSM", "BSM")
                    
                    print("  成功将 'BSM' 字段转换为 LONG。")

                elif bsm_field:
                    print("  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: " + bsm_field.type + ")")
                else:
                    print("  跳过：未找到 'BSM' 字段。")

            # 4. 内部循环的错误处理 (处理单个文件失败)
            # *** Python 3 异常处理语法 ***
            except Exception as e:
                print("  *** 错误 ***: 处理文件 " + shp_path + " 失败。")
                print("  错误信息: " + str(e))
                if arcpy.GetMessages(2):
                    print(arcpy.GetMessages(2))
                    
# 5. 外部循环的错误处理 (遍历目录失败，例如权限不足)
except Exception as e:
    print("遍历目录时发生严重错误: " + str(e))
    if arcpy.GetMessages(2):
        print(arcpy.GetMessages(2))

print("---------------------------------")
print("脚本执行完毕。")

脚本开始...
将在以下目录中搜索SHP文件 (包括所有子目录): D:\510182彭州市\510182彭州市
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\1行政区\XZQ.shp
  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: Integer)
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\2地籍区\DJQ.shp
  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: Integer)
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\3地籍子区\DJZQ.shp
  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: Integer)
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\4宅基地使用权和房屋所有权\02已调查未登记\01空间数据\JZD_ZJDSYQ_YDCWDJ.shp
  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: Integer)
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\4宅基地使用权和房屋所有权\02已调查未登记\01空间数据\JZX_ZJDSYQ_YDCWDJ.shp
  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: Integer)
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\4宅基地使用权和房屋所有权\02已调查未登记\01空间数据\ZD_ZJDSYQ_YDCWDJ.shp
  跳过：'BSM' 字段存在，但类型不是 'Double' (类型为: Integer)
---------------------------------
正在处理: D:\510182彭州市\510182彭州市\4宅基地使用权和房屋所有权\02已调查未登记\01空间数据\